In [2]:
# task 1
from fractions import Fraction

total_cards = 52
red_cards = 26
hearts = 13
diamonds = 13
face_cards = 12
diamond_face_cards = 3
spade_face_cards = 3
queen_cards = 4

p_red = Fraction(red_cards, total_cards)
print(f"P(Red) = {p_red} = {float(p_red):.4f}")

p_heart_given_red = Fraction(hearts, red_cards)
print(f"P(Heart | Red) = {p_heart_given_red} = {float(p_heart_given_red):.4f}")

p_diamond_given_face = Fraction(diamond_face_cards, face_cards)
print(f"P(Diamond | Face) = {p_diamond_given_face} = {float(p_diamond_given_face):.4f}")

spade_or_queen_face = spade_face_cards + queen_cards - 1
p_spade_or_queen_given_face = Fraction(spade_or_queen_face, face_cards)
print(f"P(Spade or Queen | Face) = {p_spade_or_queen_given_face} = {float(p_spade_or_queen_given_face):.4f}")

P(Red) = 1/2 = 0.5000
P(Heart | Red) = 1/2 = 0.5000
P(Diamond | Face) = 1/4 = 0.2500
P(Spade or Queen | Face) = 1/2 = 0.5000


In [ ]:
# task 2

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Intelligence', 'Grade'),
    ('StudyHours', 'Grade'),
    ('Difficulty', 'Grade'),
    ('Grade', 'Pass')
])

cpd_intelligence = TabularCPD('Intelligence', 2, [[0.3], [0.7]])

cpd_studyhours = TabularCPD('StudyHours', 2, [[0.4], [0.6]])

cpd_difficulty = TabularCPD('Difficulty', 2, [[0.4], [0.6]])

cpd_grade = TabularCPD(
    variable='Grade',
    variable_card=3,
    values=[
        [0.05, 0.10, 0.15, 0.20, 0.25, 0.35, 0.40, 0.50],
        [0.25, 0.30, 0.35, 0.40, 0.45, 0.40, 0.40, 0.35],
        [0.70, 0.60, 0.50, 0.40, 0.30, 0.25, 0.20, 0.15]
    ],
    evidence=['Intelligence', 'StudyHours', 'Difficulty'],
    evidence_card=[2, 2, 2]
)

cpd_pass = TabularCPD(
    variable='Pass',
    variable_card=2,
    values=[
        [0.05, 0.20, 0.50],
        [0.95, 0.80, 0.50]
    ],
    evidence=['Grade'],
    evidence_card=[3]
)

model.add_cpds(cpd_intelligence, cpd_studyhours, cpd_difficulty, cpd_grade, cpd_pass)
assert model.check_model()

inference = VariableElimination(model)

result1 = inference.query(
    variables=['Pass'],
    evidence={'StudyHours': 1, 'Difficulty': 0}
)
print("P(Pass | StudyHours=Sufficient, Difficulty=Hard):")
print(result1)

result2 = inference.query(
    variables=['Intelligence'],
    evidence={'Pass': 1}
)
print("P(Intelligence | Pass=Yes):")
print(result2)

In [ ]:
# task 3

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Disease', 'Fever'),
    ('Disease', 'Cough'),
    ('Disease', 'Fatigue'),
    ('Disease', 'Chills')
])

cpd_disease = TabularCPD('Disease', 2, [[0.3], [0.7]])

cpd_fever = TabularCPD(
    variable='Fever',
    variable_card=2,
    values=[
        [0.1, 0.5],
        [0.9, 0.5]
    ],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_cough = TabularCPD(
    variable='Cough',
    variable_card=2,
    values=[
        [0.2, 0.4],
        [0.8, 0.6]
    ],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_fatigue = TabularCPD(
    variable='Fatigue',
    variable_card=2,
    values=[
        [0.3, 0.7],
        [0.7, 0.3]
    ],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_chills = TabularCPD(
    variable='Chills',
    variable_card=2,
    values=[
        [0.4, 0.6],
        [0.6, 0.4]
    ],
    evidence=['Disease'],
    evidence_card=[2]
)

model.add_cpds(cpd_disease, cpd_fever, cpd_cough, cpd_fatigue, cpd_chills)
assert model.check_model()

inference = VariableElimination(model)

result1 = inference.query(
    variables=['Disease'],
    evidence={'Fever': 1, 'Cough': 1}
)
print("P(Disease | Fever=Yes, Cough=Yes):")
print(result1)

result2 = inference.query(
    variables=['Disease'],
    evidence={'Fever': 1, 'Cough': 1, 'Chills': 1}
)
print("P(Disease | Fever=Yes, Cough=Yes, Chills=Yes):")
print(result2)

result3 = inference.query(
    variables=['Fatigue'],
    evidence={'Disease': 0}
)
print("P(Fatigue | Disease=Flu):")
print(result3)

In [5]:
# task 4
import numpy as np
from itertools import product

states = ["Sunny", "Cloudy", "Rainy"]
state_index = {s: i for i, s in enumerate(states)}

transition_matrix = np.array([
    [0.6, 0.3, 0.1],
    [0.3, 0.4, 0.3],
    [0.2, 0.3, 0.5]
])

def simulate_weather(initial_state, num_steps):
    current = state_index[initial_state]
    sequence = [initial_state]
    for _ in range(num_steps):
        current = np.random.choice(len(states), p=transition_matrix[current])
        sequence.append(states[current])
    return sequence

np.random.seed(42)
sequence = simulate_weather("Sunny", 10)
print("Simulated weather sequence:")
print(" -> ".join(sequence))

def prob_at_least_3_rainy(initial_state, num_steps, min_rainy=3):
    num_simulations = 100000
    count = 0
    start = state_index[initial_state]
    for _ in range(num_simulations):
        current = start
        rainy_days = 0
        for _ in range(num_steps):
            current = np.random.choice(len(states), p=transition_matrix[current])
            if states[current] == "Rainy":
                rainy_days += 1
        if rainy_days >= min_rainy:
            count += 1
    return count / num_simulations

prob = prob_at_least_3_rainy("Sunny", 10)
print(f"\nP(at least 3 rainy days in 10 days | Start=Sunny) ≈ {prob:.4f}")

Simulated weather sequence:
Sunny -> Sunny -> Rainy -> Rainy -> Rainy -> Sunny -> Sunny -> Sunny -> Cloudy -> Cloudy -> Rainy

P(at least 3 rainy days in 10 days | Start=Sunny) ≈ 0.4494
